# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance"
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import dataset

# 1. Đọc và kiểm tra phân chia dữ liệu fold 0 (S1-S6)
train_df, val_df, test_df = dataset.load_split(LABELS_DIR, fold=0)
stats = dataset.check_split(train_df, val_df, test_df, IMAGES_DIR)
print('Kiểm tra tập chia S1-S6 thành công!')
print(f"Số ảnh: Train = {stats['n']['train']}, Val = {stats['n']['val']}, Test = {stats['n']['test']}, Tổng = {stats['n']['total']}")

# 2. Vẽ biểu đồ phân bố lớp
fig, ax = plt.subplots(figsize=(12, 5))
classes = [stats['per_class'][c]['name'] for c in range(9)]
train_counts = [stats['per_class'][c]['train'] for c in range(9)]
val_counts = [stats['per_class'][c]['val'] for c in range(9)]
test_counts = [stats['per_class'][c]['test'] for c in range(9)]

x = np.arange(len(classes))
w = 0.25
ax.bar(x - w, train_counts, width=w, label='Train', color='#1f77b4')
ax.bar(x, val_counts, width=w, label='Val', color='#ff7f0e')
ax.bar(x + w, test_counts, width=w, label='Test', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=35, ha='right')
ax.set_ylabel('Số lượng ảnh')
ax.set_title('Phân bố số lượng ảnh theo từng lớp trên DeepWeeds (Fold 0)')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 3. Xem trực quan 3 ảnh mẫu cho mỗi lớp
fig, axes = plt.subplots(3, 9, figsize=(18, 6))
for c_idx in range(9):
    sample_fns = train_df[train_df['Label'] == c_idx]['Filename'].head(3).tolist()
    for row_idx, fn in enumerate(sample_fns):
        img = Image.open(Path(IMAGES_DIR) / fn).convert('RGB')
        axes[row_idx, c_idx].imshow(img)
        axes[row_idx, c_idx].axis('off')
        if row_idx == 0:
            axes[row_idx, c_idx].set_title(classes[c_idx], fontsize=9, pad=3)
plt.suptitle('Ảnh mẫu đại diện 9 lớp trên DeepWeeds', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
import model, losses, train
import torch.nn as nn

# 1. Cố định seed
train.set_seed(0)

# 2. Mất mát ban đầu: xấp xỉ -ln(1/9) ≈ 2.197
test_m = model.build_model('resnet50', pretrained=False, num_classes=9)
ce_crit = nn.CrossEntropyLoss()
with torch.no_grad():
    dummy_x = torch.randn(16, 3, 224, 224)
    dummy_y = torch.randint(0, 9, (16,))
    init_loss = ce_crit(test_m(dummy_x), dummy_y).item()
print(f"Initial loss của head 9 lớp ngẫu nhiên: {init_loss:.4f} (lý thuyết ≈ {-np.log(1/9):.4f})")
assert abs(init_loss - (-np.log(1/9))) < 0.5, 'Loss ban đầu lệch quá nhiều!'

# 3. Quá khớp 1 batch nhỏ tới loss gần 0
opt = torch.optim.Adam(test_m.parameters(), lr=1e-3)
x_small = torch.randn(8, 3, 224, 224)
y_small = torch.randint(0, 9, (8,))
for it in range(80):
    opt.zero_grad()
    l = ce_crit(test_m(x_small), y_small)
    l.backward()
    opt.step()
print(f"Overfit 1 batch nhỏ sau 80 bước: loss = {l.item():.6f} (đạt chuẩn < 0.05)")
assert l.item() < 0.05, 'Không thể overfit một batch nhỏ!'

# 4. Kiểm tra ảnh sau Augmentation (giải chuẩn hóa)
train_tf = dataset.build_transforms(train=True, img_size=224, aug='color')
sample_ds = dataset.DeepWeedsDataset(train_df.head(4), IMAGES_DIR, transform=train_tf)
mean = np.array(dataset.IMAGENET_MEAN).reshape(1, 1, 3)
std = np.array(dataset.IMAGENET_STD).reshape(1, 1, 3)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    t_img, lab, fn = sample_ds[i]
    img_np = t_img.permute(1, 2, 0).numpy()
    img_unnorm = np.clip(img_np * std + mean, 0, 1)
    axes[i].imshow(img_unnorm)
    axes[i].set_title(f"{classes[lab]}\n{fn}", fontsize=8)
    axes[i].axis('off')
plt.suptitle('Ảnh sau Data Augmentation và Giải chuẩn hóa', fontsize=12)
plt.tight_layout()
plt.show()
print('Pipeline sanity checks PASSED!')


## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
from train import Config, run
import benchmark, model as model_lib

backbone_list = [
    ('B01', 'resnet50', 'ResNet-50 (Mốc chuẩn)'),
    ('B02', 'convnext_tiny', 'ConvNeXt-Tiny (CNN hiện đại)'),
    ('B03', 'resnext50_32x4d', 'ResNeXt-50 (Cardinality)'),
    ('B04', 'swin_tiny_patch4_window7_224', 'Swin-Tiny (ViT/Window Attention)'),
    ('B05', 'efficientnet_b0', 'EfficientNet-B0 (Mạng nhẹ)'),
]

bb_results = []
for exp_id, name, desc in backbone_list:
    print(f"\n=== Chạy {exp_id}: {name} ({desc}) ===")
    cfg = Config(exp_id=exp_id, backbone=name, epochs=10, batch_size=64, seed=0)
    res = run(cfg)
    m_bench = model_lib.build_model(name, pretrained=False, num_classes=9)
    lat = benchmark.latency_report(m_bench, batch_size=1, img_size=224, dtype='amp', device='cuda')
    bb_results.append({
        'exp_id': exp_id, 'backbone': name, 'tag': res['tag'],
        'params_m': res['params_m'], 'gmacs': res['gmacs'],
        'val_macro_f1': res['best_val_macro_f1'], 'val_top1': res['best_val_top1'],
        'time_per_epoch_s': res['avg_epoch_time_s'], 'latency_p50_ms': lat['p50'],
        'notes': desc
    })

df_backbones = pd.DataFrame(bb_results)
display(df_backbones)


## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# Thử nghiệm các trục công thức huấn luyện trên backbone tốt nhất (ConvNeXt-Tiny)
recipe_experiments = [
    ('T00', 'Baseline', 'Công thức nền T00', {}),
    ('T01', 'A. Khởi tạo', 'Đóng băng backbone (frozen)', {'init': 'frozen'}),
    ('T02', 'A. Khởi tạo', 'Từ đầu (scratch)', {'init': 'scratch'}),
    ('T03', 'B. Augmentation', 'ColorJitter (+ color)', {'aug': 'color'}),
    ('T04', 'B. Augmentation', 'CutMix (alpha=1.0)', {'mix': 'cutmix', 'mix_alpha': 1.0}),
    ('T05', 'B. Augmentation', 'RandAugment (randaug)', {'aug': 'randaug'}),
    ('T06', 'C. Loss', 'Label Smoothing (eps=0.1)', {'loss': 'ls', 'label_smoothing': 0.1}),
    ('T07', 'C. Loss', 'Focal Loss (gamma=2.0)', {'loss': 'focal', 'focal_gamma': 2.0}),
    ('T08', 'C. Loss', 'Class-weighted CE', {'loss': 'ce_weighted', 'class_weight_beta': 0.0}),
    ('T09', 'D. Sampler', 'WeightedRandomSampler', {'sampler': 'balanced'}),
    ('T10', 'F. Chính quy', 'EMA (decay=0.999)', {'ema_decay': 0.999}),
    ('T11', 'Combined', 'CutMix + LS + EMA', {'mix': 'cutmix', 'mix_alpha': 1.0, 'loss': 'ls', 'label_smoothing': 0.1, 'ema_decay': 0.999}),
]

train_results = []
base_f1 = None
for exp_id, axis, diff, kw in recipe_experiments:
    print(f"\n=== Chạy {exp_id} [{axis}]: {diff} ===")
    cfg = Config(exp_id=exp_id, backbone='convnext_tiny', epochs=10, batch_size=64, seed=0, **kw)
    res = run(cfg)
    if exp_id == 'T00': base_f1 = res['best_val_macro_f1']
    delta = res['best_val_macro_f1'] - base_f1
    train_results.append({
        'exp_id': exp_id, 'backbone': 'convnext_tiny', 'axis': axis,
        'diff_from_t00': diff, 'seed': 0, 'val_macro_f1': res['best_val_macro_f1'],
        'val_top1': res['best_val_top1'], 'delta_f1': round(delta, 4)
    })

df_training = pd.DataFrame(train_results)
display(df_training)


## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
import inference, benchmark, eval as ev

# Nạp validation loader và checkpoint tốt nhất (T11)
val_tf = dataset.build_transforms(train=False, img_size=224)
val_loader = dataset.make_loader(val_df, IMAGES_DIR, val_tf, batch_size=64, train=False)

best_m = model_lib.build_model('convnext_tiny', pretrained=False, num_classes=9).to(device)
best_ckpt = torch.load('runs/T11/seed0/best_model.pt', map_location=device)
best_m.load_state_dict(best_ckpt['model_state'])
best_m.eval()

# 1. I00: 1-View Baseline
names, y_val, logits_orig = inference.predict_logits(best_m, val_loader, device)
probs_orig = inference.softmax_np(logits_orig)
m_i00 = ev.compute_metrics(y_val, probs_orig.argmax(1), probs_orig)

# 2. I01: TTA Lật ngang (K=2)
_, _, logits_flip = inference.predict_logits(best_m, val_loader, device, view=inference.view_hflip)
probs_tta = inference.aggregate_views([logits_orig, logits_flip], space='prob')
m_i01 = ev.compute_metrics(y_val, probs_tta.argmax(1), probs_tta)

# 3. I07: Temperature Scaling & ECE
T_opt = inference.fit_temperature(logits_orig, y_val)
probs_cal = inference.apply_temperature(logits_orig, T_opt)
m_i07 = ev.compute_metrics(y_val, probs_cal.argmax(1), probs_cal)

print(f"I00 (1-view) Macro-F1: {m_i00['macro_f1']:.4f}, ECE: {m_i00['ece']:.4f}")
print(f"I01 (TTA K=2) Macro-F1: {m_i01['macro_f1']:.4f}, ECE: {m_i01['ece']:.4f}")
print(f"I07 (Temp Scaled T={T_opt:.3f}) Macro-F1: {m_i07['macro_f1']:.4f}, ECE: {m_i07['ece']:.4f} (ECE giảm {m_i00['ece'] - m_i07['ece']:.4f})")


## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# Chạy cấu hình chung kết F01 và mốc T00 qua 3 seed (0, 1, 2)
for seed in (0, 1, 2):
    print(f"\n--- Chạy seed {seed} ---")
    # F01: ConvNeXt-Tiny + CutMix + LS + EMA
    run(Config(exp_id='F01', backbone='convnext_tiny', seed=seed, epochs=10, batch_size=64,
               mix='cutmix', mix_alpha=1.0, loss='ls', label_smoothing=0.1, ema_decay=0.999,
               save_test_predictions=True))
    # T00: ResNet-50 Baseline
    run(Config(exp_id='T00', backbone='resnet50', seed=seed, epochs=10, batch_size=64,
               save_test_predictions=True))


In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# Xuất file kết quả tổng hợp results.xlsx theo đúng 7 sheets quy định của GUIDE.md mục 6.1
from run_experiments import step5_excel
step5_excel()
print('results.xlsx đã được cập nhật hoàn chỉnh!')
